# litterbug — Colab walkthrough

End-to-end run of the waste detection and instance-segmentation pipeline: install the project as a
library, prepare the dataset, run the exploratory analysis, fine-tune, evaluate on `valid` and on the
held-out `test` split, analyse errors, and run inference on an image and a video.

**Requirements**

- **A GPU runtime.** `Runtime -> Change runtime type -> T4 GPU`. Training refuses to start on CPU.
- **The dataset.** BUU Waste Occlusion, licensed CC BY 4.0. It is not redistributed here — step 2
  fetches it from the source.

**Expected cost:** roughly 3 hours for the two 100-epoch runs. Set `RUN_TRAINING = False` in step 4
to skip training and work with the checkpoints already in the repository.

## 1. Check the GPU

A missing GPU does not fail loudly on its own — Ultralytics will fall back to CPU and turn a
three-hour job into several days. Confirm before going further.

In [ ]:
import torch

print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("vram:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    raise SystemExit("No GPU — set Runtime -> Change runtime type -> T4 GPU and re-run.")

## 2. Install the project

Set `REPO_URL` to the repository before running.

The install is **editable, from a clone**, and that detail matters. `PROJECT_ROOT` in
`src/common/constants.py` is derived from the module's own location:

```python
PROJECT_ROOT = Path(__file__).resolve().parents[2]
DATA_YAML    = PROJECT_ROOT / "dataset" / "1_Model_Training_Data" / "data.yaml"
RUNS_DIR     = PROJECT_ROOT / "runs"
```

A plain `pip install git+...` would put `constants.py` inside `site-packages`, so `dataset/` and
`runs/` would resolve there — where the data is not. An editable install keeps the package pointing
at this clone, and everything resolves correctly.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/OWNER/litterbug.git"  # <-- edit this
WORKDIR = "/content/litterbug"

if not os.path.isdir(WORKDIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORKDIR], check=True)
os.chdir(WORKDIR)
print("working directory:", os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print("installed")

## 3. Verify the install

Import the modules and confirm the paths resolve inside the clone rather than into `site-packages`.

In [ ]:
from common.constants import CLASS_NAMES, DATA_YAML, PROJECT_ROOT, RUNS_DIR

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_YAML    :", DATA_YAML, "| exists:", DATA_YAML.is_file())
print("RUNS_DIR     :", RUNS_DIR)
print("classes      :", CLASS_NAMES)

assert "site-packages" not in str(PROJECT_ROOT), (
    "PROJECT_ROOT points into site-packages - the install was not editable. "
    "Re-run the previous cell."
)

The console script should work too. `--dry-run` resolves and prints the whole configuration, and the
environment block, without spending a second of GPU time.

In [ ]:
!litterbug train --task segment --dry-run

## 4. Get the dataset

BUU Waste Occlusion Dataset (BUU-WOD), VisionLab, Burapha University — **CC BY 4.0**, so attribution
is a licence condition, not a courtesy. The full citation is in `report/report.md` §11.

Two routes, both producing the same 2,000 images at 640×640:

- **Kaggle** — <https://www.kaggle.com/datasets/visionlab1buu/buu-waste-occlusion-dataset>. Needs an
  API token (Kaggle account → *Create New API Token*). The cell below uploads it interactively; the
  token is yours and is never written into the repository.
- **Roboflow Universe** — the export this repository was built from. See
  `dataset/1_Model_Training_Data/README.dataset.txt` for the workspace and project identifiers.

If the data is already present (for example a copy on Drive), the cell finds it and skips the
download.

In [ ]:
from pathlib import Path

DATA_DIR = Path(WORKDIR) / "dataset" / "1_Model_Training_Data"
ARCHIVE = Path("/content/kaggle")

if (DATA_DIR / "data.yaml").is_file():
    print("Dataset already in place:", DATA_DIR)
else:
    token = Path.home() / ".kaggle" / "kaggle.json"
    if not token.is_file():
        from google.colab import files

        print("Upload kaggle.json (Kaggle -> Account -> Create New API Token)")
        uploaded = files.upload()
        token.parent.mkdir(parents=True, exist_ok=True)
        token.write_bytes(next(iter(uploaded.values())))
        token.chmod(0o600)

    ARCHIVE.mkdir(parents=True, exist_ok=True)
    !pip install -q kaggle
    !kaggle datasets download -d visionlab1buu/buu-waste-occlusion-dataset -p {ARCHIVE} --unzip

    # The archive layout is the provider's, not ours. Locate data.yaml wherever it landed and link
    # it into the path the pipeline expects, rather than assuming a directory name.
    found = [p.parent for p in ARCHIVE.rglob("data.yaml")]
    if not found:
        raise FileNotFoundError(
            f"No data.yaml under {ARCHIVE}. Check what the archive contained and arrange it as"
            f" {DATA_DIR} before continuing."
        )
    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
    if not DATA_DIR.exists():
        DATA_DIR.symlink_to(found[0])
    print("linked", DATA_DIR, "->", found[0])

Validate before training. This checks image/label pairing, polygon validity, coordinate ranges and
class balance, and should reproduce the baseline counts exactly: 1,400 / 400 / 200 images and
17,400 / 4,941 / 2,507 instances.

In [ ]:
!litterbug validate --data-yaml dataset/1_Model_Training_Data/data.yaml

## 5. Exploratory analysis

Per-class counts, imbalance, geometry and lighting — read from the annotations and the pixels rather
than transcribed.

In [ ]:
from training.eda import EdaConfig, eda

dataset_eda = eda(EdaConfig())

print()
for name, row in dataset_eda["splits"].items():
    stats = row["image_stats"]
    print(
        f"{name:<6} {row['images']:>5} images  {row['instances']:>6} instances  "
        f"imbalance {row['imbalance_ratio']}  "
        f"brightness p10-p90 {stats['brightness_p10']}-{stats['brightness_p90']}"
    )

## 6. Fine-tune

Both models are trained on the same fixed schedule and the same seed, so the only thing that differs
between them is the head. That is what makes §6 of the report a comparison rather than two recipes.

**Do not read quality from a short run.** A 1-epoch run is not a preview of a 100-epoch one: `warmup_epochs=3`
and `close_mosaic=10` are absolute epoch counts, so a one-epoch schedule never finishes warmup and
never enables mosaic. It is a valid plumbing check and nothing more. Set `RUN_TRAINING = False` to
skip this entirely and use the checkpoints already in the repository.

In [ ]:
RUN_TRAINING = True  # set False to skip straight to evaluation

if RUN_TRAINING:
    !litterbug train --task segment --name litterbug-segment-yolo26s-seg
    !litterbug train --task detect --name litterbug-detect-yolo26s

# Plumbing check only - NOT a preview of quality. Its metrics must not be reported.
# !litterbug train --task segment --epochs 1 --name smoke-test

## 7. Evaluate

`valid` is for iteration. `test` is touched **once**, and only after every model decision is final —
otherwise the reported numbers become the ones the model was selected on.

In [ ]:
from pathlib import Path


def newest_checkpoint(task: str) -> Path:
    """Most recent best.pt for a task, so the notebook does not hardcode a run name."""
    candidates = sorted(
        Path(RUNS_DIR).glob(f"*{task}*/weights/best.pt"), key=lambda p: p.stat().st_mtime
    )
    if not candidates:
        raise FileNotFoundError(f"no checkpoint under {RUNS_DIR} for {task!r}")
    return candidates[-1]


SEGMENT = newest_checkpoint("seg")
DETECT = newest_checkpoint("detect")
print("segment:", SEGMENT)
print("detect :", DETECT)

In [ ]:
!litterbug val --weights {SEGMENT} --split val

In [ ]:
# The held-out split. Run once.
!litterbug val --weights {SEGMENT} --split test
!litterbug val --weights {DETECT} --split test

## 8. Error analysis

Ultralytics computes per-instance matches internally but does not expose them, so this re-derives the
matching from scratch: greedy, one-to-one, by descending confidence, class-agnostic, at mask IoU 0.5.

The `--iou-mode box` run goes to a separate directory — the split and the matching mode are part of the
run name precisely so one analysis cannot silently overwrite another.

In [ ]:
from training.error_analysis import AnalysisConfig, analyse

mask_analysis = analyse(AnalysisConfig(weights=SEGMENT, split="test"))
box_analysis = analyse(AnalysisConfig(weights=SEGMENT, split="test", iou_mode="box"))

The commentary the figures carry is measured, not interpreted: instance counts, class composition, the
size quartile of each miss, the confidences of the unmatched predictions. Causes are left to the
report.

In [ ]:
from IPython.display import Image, Markdown, display
from training.error_examples import ExampleConfig, examples

gallery = examples(ExampleConfig(weights=SEGMENT, split="test"))
display(Markdown(Path(gallery["commentary"]).read_text(encoding="utf-8")[:2000]))

first = sorted(Path(gallery["figures_dir"]).glob("*.png"))[0]
display(Image(filename=str(first)))

## 9. Inference

`predict()` returns structured detections as well as writing artifacts, so the demo API can consume it
without reading files off disk.

In [ ]:
from running.predict import PredictConfig, predict

sample = sorted(Path("dataset/1_Model_Training_Data/test/images").glob("*.jpg"))[0]
result = predict(PredictConfig(weights=SEGMENT, source=sample, name="notebook-sample"))

print("kind       :", result.kind)
print("detections :", len(result.detections))
print("counts     :", result.counts)
print("artifacts  :")
for path in result.artifacts:
    print("  ", path)

### Video, with tracking

Point `VIDEO` at a clip and this links detections across frames with ByteTrack. Tracking adds a
measurement no mAP can give on unlabelled footage: how long each detection survives. Confidence rises
monotonically with persistence, so track length acts as a free reliability filter.

In [ ]:
VIDEO = None  # e.g. Path("/content/my_clip.mp4")

if VIDEO is not None:
    tracked = predict(PredictConfig(weights=SEGMENT, source=Path(VIDEO), name="notebook-video",
                                    tracker="bytetrack.yaml"))
    print("tracks:", tracked.tracks["unique_tracks"], "from", len(tracked.detections), "detections")
    print("median track length:", tracked.tracks["median_track_length_frames"], "frames")
else:
    print("Set VIDEO to a clip to run the tracked pass.")

## 10. Reference results

The values the committed checkpoints produce, for comparison against this run.

**Segmentation, `test` (200 images / 2,507 instances)**

| Metric | Box | Mask |
| --- | --- | --- |
| mAP50-95 | 0.837 | 0.796 |
| mAP50 | 0.954 | 0.955 |
| Precision | 0.939 | 0.944 |
| Recall | 0.901 | 0.905 |

Mean IoU over matched pairs: 0.901 mask, 0.924 box.

**Detection baseline, `test`:** box mAP50-95 0.8376, mAP50 0.9557, P 0.944, R 0.913.

The box-head difference between the two models is 0.0004 — nothing. Segmentation delivers masks while
giving up nothing measurable in detection quality.

`valid` runs one to two points higher (box 0.857 / mask 0.814 mAP50-95), and nearly all of that gap
sits in the smallest size quartile — which is the same small-object weakness the error analysis finds
on `valid`, amplified on unseen data.